# Tagging - Interpretando dados com funções

Uma das principais aplicações das funções externas por desenvolvedores não é exatamente na utilização dessas funções e sim, na utilização desta sintaxe e estruturação de dados gerado pela API da OpenAI para a categorização e estruturação de dados em texto. Daremos um exemplo aqui ao categorizanto sentimentos de falas.

In [16]:

# Importar Bibliotecas Necessárias
import os
from dotenv import load_dotenv
import requests
from langchain_community.llms import Ollama

# Carregar variáveis de ambiente
load_dotenv()

print("✅ Bibliotecas importadas com sucesso!!!! ")


✅ Bibliotecas importadas com sucesso!!!! 


In [19]:
# Verificar conexão com Ollama
OLLAMA_BASE_URL = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
    if response.status_code == 200:
        modelos = response.json()
        print("✅ Conexão com Ollama estabelecida!")
        print(f"\n📦 Modelos disponíveis:")
        for modelo in modelos.get('models', []):
            print(f"  - {modelo['name']}")
    else:
        print("❌ Erro ao conectar com Ollama")
except requests.exceptions.ConnectionError:
    print("❌ Não foi possível conectar ao Ollama. Certifique-se que está rodando em Docker")

✅ Conexão com Ollama estabelecida!

📦 Modelos disponíveis:
  - qwen3.5:9b
  - llama3.1:8b-instruct-q4_K_M
  - nomic-embed-text:latest
  - llama3.2:3b
  - nomic-embed-text:v1.5
  - qwen3:4b
  - olmo-3:7b
  - llama3.2:1b
  - llama3.1:8b
  - llama3.2:latest


In [13]:
from langchain.tools import tool
from pydantic import BaseModel, Field #Importação atualizada
from langchain_core.utils.function_calling import convert_to_openai_function

@tool("Sentimento", return_direct=True)
class Sentimento(BaseModel):
    '''Define o sentimento e a língua da mensagem enviada'''
    sentimento: str = Field(description='Sentimendo do texto. Deve ser "pos", "neg" ou "nd" para não definido.')
    lingua: str = Field(description='Língua que o texto foi escrito (deve estar no formato ISO 639-1)')

Sentimento 

StructuredTool(name='Sentimento', description='Define o sentimento e a língua da mensagem enviada', args_schema=<class 'langchain_core.utils.pydantic.Sentimento'>, return_direct=True, func=<class '__main__.Sentimento'>)

In [4]:
texto = 'Eu gosto muito de massa aos quatro queijos'

In [10]:
from langchain.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama 

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Pense com cuidado ao categorizar o texto conforme as instruções'),
    ('user', '{input}')
])


chat = ChatOllama(model='qwen3.5:9b', temperature=0)

llm_with_tools = chat.bind_tools([tool_sentimento])

chain = prompt | llm_with_tools

In [31]:
from enum import Enum
from pydantic import BaseModel, Field
from langchain_core.tools import tool    

@tool
def calcular_desconto(preco: float , percentual: float ) -> float:
    """ Calcula o preço final de um produto após aplicar um desconto percentual."""
    return preco * (1 - percentual / 100 )

class Descontotag(str, Enum):
    alto = "alto"
    baixo = "baixo"
    sem = "sem desconto"

class Valor(BaseModel):
    preco : float = Field(description='Valor do produto')
    percentual: float = Field(description='Percentual de desconto a ser aplicado')  
    desconto: Descontotag = Field(description='Nível de desconto aplicado ao produto')

In [30]:
chat = ChatOllama(model="llama3.2:3B", temperature=0)


llm_com_tool = chat.bind_tools([Valor, calcular_desconto])

pergunta = "Qual o valor final de um produto que custa R$ 100,00 com um desconto de 20% ? E qual o nível de desconto aplicado?"


ai_msg = llm_com_tool.invoke(pergunta)

print(ai_msg)

content='' additional_kwargs={} response_metadata={'model': 'llama3.2:3B', 'created_at': '2026-03-27T15:41:35.2341101Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20566012800, 'load_duration': 10526964600, 'prompt_eval_count': 290, 'prompt_eval_duration': 7675839700, 'eval_count': 27, 'eval_duration': 2311739800, 'logprobs': None, 'model_name': 'llama3.2:3B'} id='run--019d2ff4-f02b-7271-8ed8-2d41d8c800dd-0' tool_calls=[{'name': 'calcular_desconto', 'args': {'percentual': '0', 'preco': '100'}, 'id': 'b547ac82-ee9c-41be-8b0e-545560822efd', 'type': 'tool_call'}] usage_metadata={'input_tokens': 290, 'output_tokens': 27, 'total_tokens': 317}


In [21]:
from enum import Enum
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

class TipoSentimento(str, Enum):
    POS = "pos"
    NEG = "neg"
    ND = "nd"

class Idioma(str, Enum):
    PT = "pt"
    EN = "en"

class Sentimento(BaseModel):
    sentimento: TipoSentimento = Field(description='Use apenas "pos", "neg" ou "nd"')
    lingua: Idioma = Field(description='Código ISO 639-1')

chat = ChatOllama(model="llama3.2:3B", temperature=0)

structured_llm = chat.with_structured_output(Sentimento)

res = structured_llm.invoke(
    "Eu gosto muito de massa aos quatro queijos"
)

print(res)

sentimento=<TipoSentimento.NEG: 'neg'> lingua=<Idioma.PT: 'pt'>


In [22]:
# chain.invoke({'input': 'Eu gosto muito de massa aos quatro queijos'})
res = structured_llm.invoke(
    "I don't like this movie at all, it was a waste of time"
)

print(res)

sentimento=<TipoSentimento.NEG: 'neg'> lingua=<Idioma.EN: 'en'>


In [5]:
chain.invoke({'input': 'I dont like this food'})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"sentimento":"neg","lingua":"en"}', 'name': 'Sentimento'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 138, 'total_tokens': 150, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-895d9189-fb30-4c70-829f-56124bca222a-0', usage_metadata={'input_tokens': 138, 'output_tokens': 12, 'total_tokens': 150, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Parseando a saída para obtermos apenas o que interessa

Podemos utilizar o JsonOutputFunctionsParser para obtermos como resultado final da nossa chain apenas o que nos interessa, que é o dicionário com as tags do conteúdo.

In [6]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser


chain = (prompt 
         | chat.bind(functions=[tool_sentimento], function_call={'name': 'Sentimento'})
         | JsonOutputFunctionsParser())
chain.invoke({'input': 'Eu gosto muito de massa aos quatro queijos'})

{'sentimento': 'pos', 'lingua': 'pt'}

In [7]:
chain.invoke({'input': 'I dont like this food'})

{'sentimento': 'neg', 'lingua': 'en'}

## Um exemplo mais interessante

Indo para um exemplo um pouco mais complexo. Digamos que temos um chatbot e queremos fazer um roteamento para os setores de interesse, no nosso caso atendimento_cliente, duvidas_alunos, vendas, spam. A primeira etapa é criar um modelo que entenda a solicitação do cliente e direcione para o setor certo. Nas próximas etapas o atendimento pode ser continuado por pessoas ou por agentes especializados para as tarefas do setor.
Retiramos as seguintes mensagens de email recebidos pela nossa equipe de atendimento e vamos tentar criar um direcionamento para elas:

In [8]:
duvidas = [
    'Bom dia, gostaria de saber se há um certificado final para cada trilha ou se os certificados são somente para os cursos e projetos? Obrigado!',
    'In Etsy, Amazon, eBay, Shopify https://pint77.com Pinterest+SEO +II = high sales results',
    'Boa tarde, estou iniciando hoje e estou perdido. Tenho vários objetivos. Não sei nada programação, exceto que utilizo o Power automate desktop da Microsoft. Quero aprender tudo na plataforma que se relacione ao Trading de criptomoedas. Quero automatizar Tradings, fazer o sistema reconhecer padrões, comprar e vender segundo critérios que eu defina, etc. Também tenho objetivos de aprender o máximo para utilizar em automações no trabalho também, que envolve a área jurídica e trabalho em processos. Como sou fã de eletrônica e tenho cursos na área, também queria aprender o que precisa para automatizacões diversas. Existe algum curso ou trilha que me prepare com base para todas essas áreas ao mesmo tempo e a partir dele eu aprenda isoladamente aquilo que seria exigido para aplicar aos meus projetos?',
    'Bom dia, Havia pedido cancelamento de minha mensalidade no mes 2 e continuaram cobrando. Peço cancelamento da assinatura. Peço por gentileza, para efetivarem o cancelamento da assomatura e pagamento.',
    'Bom dia. Não estou conseguindo tirar os certificados dos cursos que concluí. Por exemplo, já consegui 100% no python starter, porém, não consigo tirar o certificado. Como faço?',
    'Bom dia. Não enconte no site o preço de um curso avulso. SAberiam me informar?'
    ]

In [9]:
from enum import Enum
from pydantic import BaseModel, Field #Importação atualizada

class SetorEnum(str, Enum):
    atendimento_cliente = 'atendimento_cliente'
    duvidas_aluno = 'duvidas_aluno'
    vendas = 'vendas'
    spam = 'spam'

class DirecionaSetorResponsavel(BaseModel):
    """Direciona a dúvida de um cliente ou aluno da escola de programação Asimov para o setor responsável"""
    setor: SetorEnum

In [10]:
from langchain_core.utils.function_calling import convert_to_openai_function

tool_direcionamento = convert_to_openai_function(DirecionaSetorResponsavel)
tool_direcionamento

{'name': 'DirecionaSetorResponsavel',
 'description': 'Direciona a dúvida de um cliente ou aluno da escola de programação Asimov para o setor responsável',
 'parameters': {'properties': {'setor': {'enum': ['atendimento_cliente',
     'duvidas_aluno',
     'vendas',
     'spam'],
    'type': 'string'}},
  'required': ['setor'],
  'type': 'object'}}

In [11]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Pense com cuidado ao categorizar o texto conforme as instruções'),
    ('user', '{input}')
])
chat = ChatOpenAI()

chain = (prompt 
         | chat.bind(functions=[tool_direcionamento], function_call={'name': 'DirecionaSetorResponsavel'})
         | JsonOutputFunctionsParser())

In [12]:
duvida = duvidas[5]
resposta = chain.invoke({'input': duvida})
print('Dúvida:', duvida)
print('Resposta:', resposta)

Dúvida: Bom dia. Não enconte no site o preço de um curso avulso. SAberiam me informar?
Resposta: {'setor': 'atendimento_cliente'}


Neste caso, gostaríamos que a dúvida fosse direcionada para vendas. Podemos melhorar nosso prompt para auemntar a chance do modelo responder como gostaríamos:

In [13]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

system_message = '''Pense com cuidado ao categorizar o texto conforme as instruções.
Questões relacionadas a dúvidas de preço, sobre o produto, como funciona devem ser direciodas para "vendas".
Questões relacionadas a conta, acesso a plataforma, a cancelamento e renovação de assinatura para devem ser direciodas para "atendimento_cliente".
Questões relacionadas a dúvidas técnicas de programação, conteúdos da plataforma ou tecnologias na área da programação devem ser direciodas para "duvidas_alunos".
Mensagens suspeitas, em outras línguas que não português, contendo links devem ser direciodas para "spam".
'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_message),
    ('user', '{input}')
])
chat = ChatOpenAI()

chain = (prompt 
         | chat.bind(functions=[tool_direcionamento], function_call={'name': 'DirecionaSetorResponsavel'})
         | JsonOutputFunctionsParser())

In [14]:
duvida = duvidas[5]
resposta = chain.invoke({'input': duvida})
print('Dúvida:', duvida)
print('Resposta:', resposta)

Dúvida: Bom dia. Não enconte no site o preço de um curso avulso. SAberiam me informar?
Resposta: {'setor': 'vendas'}
